# Fraud Compliance Agent Notebook 03 — Plaid-to-canonical mapping

**Fraud Compliance Agent · P0-03 — proposed mapping review**  
**Status:** Proposal execution complete — review pending  
**Decision supported:** P0-03 — proposed mapping review

---

<a id="plain-english"></a>
## In plain English

This notebook is a **translation check**, not a fraud model. Plaid can describe a bank transaction one way, while our app needs one consistent format before it can safely compare transactions, run controls, or show an audit trail.

Think of it like agreeing how to label the columns when combining two spreadsheets. For example, it checks whether Plaid gives us a usable date, amount, currency, and direction (money in or money out), and records exactly how each one would be represented in our proposed app format.

It does **not** connect to Plaid, make a payment decision, train a model, or approve the proposed data format. Where the information is uncertain or incomplete, the notebook deliberately labels it as a blocker rather than guessing. The final output is therefore a small review pack for a human to approve, reject, or improve.

### How to read it

1. **Step 1** checks that the earlier, sanitised evidence exists.
2. **Step 2** lists the proposed translation for each field.
3. **Step 3** tests safe made-up examples, so no real bank data is needed.
4. **Step 4** produces a review-only summary and checksums, so reviewers can tell which files were checked.

<a id="purpose"></a>
## Purpose

**Question:** How does each observed field map to the proposed SourceEvent and CanonicalTransaction draft?

Turn evidence from the inventory and lifecycle probes into a reviewable mapping; it does not accept a contract.

### Non-goals

- No production ingestion, policy, contract, or model release is approved by this notebook.
- No raw provider payloads, identifiers, credentials, customer data, model weights, or hidden reasoning may enter Git.
- Missing evidence remains unknown; it must never become a fabricated value or result.

<a id="contents"></a>
## Contents

1. [In plain English](#plain-english)
2. [Purpose](#purpose)
3. [Pre-flight and safety](#pre-flight)
4. [Load reviewed source evidence](#step-1)
5. [Build a field-by-field candidate mapping](#step-2)
6. [See a Plaid example safely](#safe-preview)
7. [Test safe example shapes](#step-3)
8. [Publish a mapping proposal](#step-4)
9. [Findings, limitations, and next gate](#review)

---

<a id="pre-flight"></a>
## Pre-flight and safety

Run from the repository with the **Fraud Compliance Agent API (Python 3.11)** kernel. List environment variables by name only. Clear every output before committing.

**Expected sanitised artifact:** `docs/proposals/plaid-to-canonical-mapping.proposed.json`


In [ ]:
from __future__ import annotations

import json
import subprocess
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd

# Find the repository root so all artifact paths remain portable.
REPOSITORY_ROOT = Path.cwd().resolve()
while REPOSITORY_ROOT != REPOSITORY_ROOT.parent and not (REPOSITORY_ROOT / "AGENTS.md").exists():
    REPOSITORY_ROOT = REPOSITORY_ROOT.parent

if not (REPOSITORY_ROOT / "AGENTS.md").exists():
    raise RuntimeError("Run this notebook from inside the fraud-compliance-agent repository.")

def git_revision() -> str:
    """Return the current commit when available without failing an exploratory run."""
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=REPOSITORY_ROOT, text=True, stderr=subprocess.DEVNULL
        ).strip()
    except (OSError, subprocess.CalledProcessError):
        return "uncommitted-or-unavailable"

# Capture only reproducibility metadata; never capture credentials or source records.
RUN_CONTEXT = {
    "run_at_utc": datetime.now(UTC).isoformat(),
    "git_revision": git_revision(),
    "notebook_status": "proposal-executed-review-pending",
}
print("Repository:", REPOSITORY_ROOT)
print("Git revision:", RUN_CONTEXT["git_revision"])
print("Safety: do not print secrets, raw provider payloads, identifiers, or model artifacts.")


<a id="step-1"></a>
## Step 1 — Load reviewed source evidence

### What this notebook does

Classify every canonical field as observed Plaid field, deterministic derivation, synthetic scenario-only value, or unavailable.

### Safe success condition

The result is an explicit, sanitised observation or a stated blocker. It is never an implicit contract approval, production integration, or model promotion.

### Code walkthrough

The next cell checks that the reviewed inventory and canonical draft exist. It reads file paths only at this stage; it does not call Plaid or load a raw provider response.


In [ ]:
from pathlib import Path

# Declare the minimum reviewed artifacts required before mapping work can start.
REQUIRED_REVIEW_INPUTS = [
  "docs/proposals/plaid-source-inventory.observation.json",
  "docs/proposals/canonical-transaction-contract.proposed.md"
]
def find_missing_review_inputs(paths: list[str]) -> list[str]:
    """Return absent reviewed artifacts without substituting fabricated evidence."""
    return [path for path in paths if not (REPOSITORY_ROOT / path).exists()]

# Stop the proposal workflow when evidence is absent rather than fabricating it.
missing = find_missing_review_inputs(REQUIRED_REVIEW_INPUTS)
if missing:
    print("GATED — this notebook has not run because required review inputs are absent:")
    for path in missing:
        print(f"- {path}")
    print("Do not substitute fabricated inputs. Record the blocker in the matching experiment record.")
else:
    print("Required review inputs are present. Continue only after confirming their approval status.")


<a id="step-2"></a>
## Step 2 — Build a field-by-field candidate mapping

### What this notebook does

State money units, direction, currency, account scope, category treatment, timestamp precedence, source revision, and missingness for every proposed mapping.

### Safe success condition

The result is an explicit, sanitised observation or a stated blocker. It is never an implicit contract approval, production integration, or model promotion.

### Code walkthrough

The next cell loads only committed sanitised artifacts, checks every mapping classification against the allowed vocabulary, and prints aggregate counts plus unresolved fields. It does not infer missing Plaid semantics.


In [ ]:
from hashlib import sha256

# Point only at committed sanitised evidence, never at a raw Plaid response.
MAPPING_PATH = REPOSITORY_ROOT / "docs/proposals/plaid-to-canonical-mapping.proposed.json"
LIFECYCLE_PATH = REPOSITORY_ROOT / "docs/proposals/plaid-lifecycle-probes.observation.json"

def load_sanitised_json(path: Path) -> dict:
    """Load a reviewed proposal; raw provider payloads are never notebook inputs."""
    if not path.is_file():
        raise FileNotFoundError(f"Missing sanitised proposal: {path.relative_to(REPOSITORY_ROOT)}")
    return json.loads(path.read_text(encoding="utf-8"))

# Load the source inventory, lifecycle evidence, and candidate mapping for review.
inventory = load_sanitised_json(REPOSITORY_ROOT / "docs/proposals/plaid-source-inventory.observation.json")
lifecycle = load_sanitised_json(LIFECYCLE_PATH)
mapping_proposal = load_sanitised_json(MAPPING_PATH)
# Restrict mapping rows to the explicit evidence vocabulary used by this project.
allowed_classifications = {
    "observed", "observed_nullable", "observed_with_date_precision",
    "observed_but_not_normalised", "deterministic_derivation",
    "candidate_derivation", "unavailable", "unavailable_by_default",
}
def validate_mapping_rows(rows: list[dict], allowed: set[str]) -> None:
    """Require non-empty mapping rows that use only the reviewed classification vocabulary."""
    if not rows or not all(row.get("classification") in allowed for row in rows):
        raise ValueError("Mapping rows must be present and use only reviewed classifications.")

mapping_rows = mapping_proposal.get("mapping", [])
validate_mapping_rows(mapping_rows, allowed_classifications)

# Ensure high-risk unknowns cannot disappear from the candidate mapping.
canonical_fields = {row["canonical_field"] for row in mapping_rows}
required_blockers = {
    "SourceEvent.observed_at and available_at",
    "CanonicalTransaction.direction",
    "CanonicalTransaction.money.amount_minor",
}
if not required_blockers.issubset(canonical_fields):
    raise ValueError("Required timing, direction, and money blockers are absent.")

# Summarise evidence status without printing individual source values.
classification_counts = {
    classification: sum(row["classification"] == classification for row in mapping_rows)
    for classification in sorted({row["classification"] for row in mapping_rows})
}
unresolved = [
    row["canonical_field"] for row in mapping_rows
    if row["classification"] in {"unavailable", "unavailable_by_default", "observed_but_not_normalised", "candidate_derivation"}
]
print("Sanitised source inventory fields:", len(inventory["field_inventory"]))
print("Candidate mapping rows:", len(mapping_rows))
print("Classification counts:", classification_counts)
print("Unresolved fields:", ", ".join(unresolved))


<a id="safe-preview"></a>
## See a Plaid example safely

The next cell turns the real **Sandbox observation** into a small, readable preview. It shows only field paths, field types, whether a field can be empty, and the proposed mapping status. It never loads or prints a transaction, account ID, merchant, amount, date, token, or any other source value.

**Inputs:** the committed sanitised field inventory and mapping proposal already loaded in Step 2.  
**Output:** a Pandas review table with five representative field mappings, their current status, and what that status means.  
**Safety boundary:** this is evidence of the observed Plaid Sandbox response shape, not an approved production connector or a complete canonical transaction.


In [ ]:
def build_safe_plaid_preview(inventory: dict, rows: list[dict]) -> list[dict[str, str]]:
    """Build a redacted mapping preview from reviewed Sandbox metadata.

    Args:
        inventory: Sanitised Plaid Sandbox field inventory containing paths, types,
            nullability, and precision metadata only; it must not contain source values.
        rows: Proposed canonical mapping rows with source paths and review classifications.

    Returns:
        Five representative, display-safe dictionaries. Each contains only field names,
        observed metadata, a proposed target, a review status, and a safe explanation.

    Raises:
        ValueError: If a representative source path is absent from the sanitised inventory
        or mapping proposal, because the preview must not fill gaps by guessing.

    Side effects:
        None. The function does not call Plaid, read credentials, persist data, or expose
        transaction values, identifiers, account details, or customer-like information.
    """
    # Index metadata by path so the preview can prove each shown field was observed.
    inventory_by_path = {field["path"]: field for field in inventory["field_inventory"]}
    mappings_by_source = {row.get("source"): row for row in rows if row.get("source")}

    # Select only representative transaction fields; this list contains no source values.
    preview_sources = [
        "$.added[].transaction_id",
        "$.added[].date",
        "$.added[].amount",
        "$.added[].iso_currency_code",
        "$.added[].payment_channel",
    ]
    preview = []
    for source_path in preview_sources:
        if source_path not in inventory_by_path or source_path not in mappings_by_source:
            raise ValueError(f"Safe preview is missing reviewed evidence for: {source_path}")
        field = inventory_by_path[source_path]
        mapping = mappings_by_source[source_path]
        preview.append({
            "Plaid Sandbox field": source_path,
            "Observed type": " / ".join(field["observed_types"]),
            "Can be empty": str(field["nullable"]),
            "Proposed app field": mapping["canonical_field"],
            "Review status": mapping["classification"],
            "What this means": mapping["handling"],
        })
    return preview

def as_safe_preview_table(preview: list[dict[str, str]]) -> pd.DataFrame:
    """Format reviewed mapping metadata as a readable Pandas table.

    Args:
        preview: Display-safe mapping dictionaries from ``build_safe_plaid_preview``.
            They may contain schema metadata and review notes, but never source values.

    Returns:
        A Pandas DataFrame with a stable, reviewer-oriented column order. The returned
        table is suitable for notebook display and contains no raw Plaid records.

    Raises:
        ValueError: If the supplied preview is empty, because an empty table could be
        mistaken for an observed result.

    Side effects:
        None. Formatting the table does not call Plaid, write files, or reveal data.
    """
    if not preview:
        raise ValueError("Safe preview must contain reviewed mapping rows.")
    return pd.DataFrame(preview)[[
        "Plaid Sandbox field",
        "Observed type",
        "Can be empty",
        "Proposed app field",
        "Review status",
        "What this means",
    ]]

# Build a readable, redacted preview from the evidence loaded in Step 2.
safe_preview = build_safe_plaid_preview(inventory, mapping_rows)
safe_preview_table = as_safe_preview_table(safe_preview)

# Display the table directly so Jupyter renders a scannable review view.
safe_preview_table


<a id="step-3"></a>
## Step 3 — Test safe example shapes

### What this notebook does

Validate a committed candidate fixture manifest containing only synthetic draft shapes and explicit Plaid blockers. Never copy raw provider values into Git.

### Safe success condition

The result is an explicit, sanitised observation or a stated blocker. It is never an implicit contract approval, production integration, or model promotion.

### Code walkthrough

The next cell verifies fixture IDs and validates that the only passing shape is explicitly synthetic. The Plaid-shaped examples remain blockers, so the notebook cannot accidentally turn them into usable source facts.


In [ ]:
# Load only the redacted candidate fixture manifest.
FIXTURE_PATH = REPOSITORY_ROOT / "docs/proposals/plaid-canonical-fixture-manifest.proposed.json"
fixture_manifest = load_sanitised_json(FIXTURE_PATH)
def fixture_ids_from(manifest: dict) -> list[str]:
    """Return fixture IDs from the sanitised manifest without exposing fixture payload values."""
    return [fixture.get("fixture_id") for fixture in manifest.get("fixtures", [])]

fixture_ids = fixture_ids_from(fixture_manifest)
# Require one synthetic schema shape and both known Plaid blocker cases.
required_fixture_ids = {
    "draft-canonical-valid-shape",
    "plaid-date-precision-blocker",
    "plaid-money-and-direction-blocker",
}
if not fixture_ids or len(fixture_ids) != len(set(fixture_ids)):
    raise ValueError("Fixture IDs must be present and unique.")
if not required_fixture_ids.issubset(set(fixture_ids)):
    raise ValueError("Fixture manifest must include the draft shape and both known Plaid blockers.")

# Verify that the only valid example is explicitly synthetic, not provider evidence.
valid_shape = next(fixture for fixture in fixture_manifest["fixtures"] if fixture["fixture_id"] == "draft-canonical-valid-shape")
if valid_shape.get("classification") != "synthetic_schema_only":
    raise ValueError("The valid-shape fixture must remain explicitly synthetic.")
if valid_shape["canonical_transaction"]["money"]["amount_minor"] < 0:
    raise ValueError("Draft canonical money must remain non-negative minor units.")

# Keep unresolved Plaid semantics visible as blockers rather than usable facts.
blocker_fixtures = [fixture for fixture in fixture_manifest["fixtures"] if fixture.get("classification") == "plaid_mapping_blocker"]
if len(blocker_fixtures) != 2:
    raise ValueError("Exactly two explicit Plaid mapping blockers are expected in this proposal.")
print("Candidate fixtures:", ", ".join(fixture_ids))
print("Synthetic valid draft shape:", valid_shape["fixture_id"])
print("Explicit Plaid blockers:", ", ".join(fixture["fixture_id"] for fixture in blocker_fixtures))


<a id="step-4"></a>
## Step 4 — Publish a mapping proposal

### What this notebook does

Validate mapping classifications, fixture IDs, blockers, and sanitised artifact digests; leave accepted docs/contracts unchanged.

### Safe success condition

The result is an explicit, sanitised observation or a stated blocker. It is never an implicit contract approval, production integration, or model promotion.

### Code walkthrough

The final cell rechecks the sanitised inputs defensively, builds a review-only report in memory, and prints artifact checksums. It does not write an accepted contract or enable a connector.


In [ ]:
# Re-load the same sanitised artifacts for a self-contained final report.
MAPPING_PATH = REPOSITORY_ROOT / "docs/proposals/plaid-to-canonical-mapping.proposed.json"
LIFECYCLE_PATH = REPOSITORY_ROOT / "docs/proposals/plaid-lifecycle-probes.observation.json"
FIXTURE_PATH = REPOSITORY_ROOT / "docs/proposals/plaid-canonical-fixture-manifest.proposed.json"

def load_sanitised_json(path: Path) -> dict:
    """Load a reviewed, committed JSON proposal; never load a raw provider payload."""
    if not path.is_file():
        raise FileNotFoundError(f"Required sanitised proposal is missing: {path.relative_to(REPOSITORY_ROOT)}")
    return json.loads(path.read_text(encoding="utf-8"))

inventory = load_sanitised_json(REPOSITORY_ROOT / "docs/proposals/plaid-source-inventory.observation.json")
lifecycle = load_sanitised_json(LIFECYCLE_PATH)
mapping_proposal = load_sanitised_json(MAPPING_PATH)
fixture_manifest = load_sanitised_json(FIXTURE_PATH)

allowed_classifications = {
    "observed", "observed_nullable", "observed_with_date_precision",
    "observed_but_not_normalised", "deterministic_derivation",
    "candidate_derivation", "unavailable", "unavailable_by_default",
}
mapping_rows = mapping_proposal.get("mapping", [])
if not mapping_rows:
    raise ValueError("Mapping proposal contains no field rows.")
if not all(row.get("classification") in allowed_classifications for row in mapping_rows):
    raise ValueError("Mapping proposal contains an unreviewed classification.")

canonical_fields = {row["canonical_field"] for row in mapping_rows}
required_blockers = {
    "SourceEvent.observed_at and available_at",
    "CanonicalTransaction.direction",
    "CanonicalTransaction.money.amount_minor",
}
if not required_blockers.issubset(canonical_fields):
    raise ValueError("Required point-in-time and money/direction blockers are absent.")

fixture_ids = [fixture.get("fixture_id") for fixture in fixture_manifest.get("fixtures", [])]
if len(fixture_ids) != len(set(fixture_ids)) or not fixture_ids:
    raise ValueError("Fixture manifest needs unique, non-empty fixture IDs.")
if "draft-canonical-valid-shape" not in fixture_ids:
    raise ValueError("Fixture manifest must include one clearly synthetic draft-shape fixture.")

classification_counts = {
    classification: sum(row["classification"] == classification for row in mapping_rows)
    for classification in sorted({row["classification"] for row in mapping_rows})
}
unresolved = [
    row["canonical_field"] for row in mapping_rows
    if row["classification"] in {"unavailable", "unavailable_by_default", "observed_but_not_normalised", "candidate_derivation"}
]

def sha256_for(path: Path) -> str:
    """Return a reproducibility checksum for a committed sanitised artifact."""
    return sha256(path.read_bytes()).hexdigest()

# Pin the review artifacts to their exact committed bytes.
artifact_paths = [MAPPING_PATH, FIXTURE_PATH]
artifact_digests = {
    str(path.relative_to(REPOSITORY_ROOT)): sha256_for(path)
    for path in artifact_paths
}
# Assemble an in-memory review summary; do not promote or write a runtime contract.
report = {
    "notebook": "03-plaid-to-canonical-mapping",
    "status": "proposal-executed-review-pending",
    "run_context": RUN_CONTEXT,
    "input_artifacts": ["plaid-source-inventory", "plaid-lifecycle-probes", "canonical-transaction-draft"],
    "mapping_row_count": len(mapping_rows),
    "classification_counts": classification_counts,
    "fixture_ids": fixture_ids,
    "unresolved_canonical_fields": unresolved,
    "lifecycle_unknowns": [
        name for name, state in lifecycle["lifecycle_observations"].items()
        if state == "indeterminate"
    ],
    "artifact_sha256": artifact_digests,
    "decision_recommendation": "proposed — no contract approval, connector, feature set, or payment action is implied",
}

print("Sanitised source inventory fields:", len(inventory["field_inventory"]))
print("Candidate mapping rows:", report["mapping_row_count"])
print("Classification counts:", classification_counts)
print("Candidate fixtures:", ", ".join(fixture_ids))
print("Unresolved fields:", ", ".join(unresolved))
print("Lifecycle unknowns:", ", ".join(report["lifecycle_unknowns"]))
print("Artifact digests:", artifact_digests)
print("Result: proposal evidence is complete; review and acceptance remain separate gates.")


<a id="review"></a>
## Findings, limitations, and next gate

- Findings: the notebook validates 15 mapping rows plus one synthetic draft-shape fixture and two explicit Plaid blocker fixtures.
- Limitations: required acceptance and several point-in-time semantics remain absent.
- Recommendation: **proposed** — do not approve a contract, feature, policy, or model from this template.
- Next gate: update `docs/experiments/03-plaid-to-canonical-mapping.md` after a run, then request the named review decision.

## Reviewer checklist

- [x] Outputs are cleared and contain no secrets, raw provider data, PII, identifiers, or hidden reasoning.
- [x] Every result is labelled observed, unsupported, indeterminate, unavailable, or proposed as appropriate.
- [x] The matching experiment record contains revision, inputs, findings, limitations, and artifact digests.
- [ ] No runtime contract, threshold, model promotion, or payment action was inferred.

